In [1]:
import os
import easyocr
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# Initialize the OCR reader
# en for English - you can add more languages as needed
reader = easyocr.Reader(['en'])

# Path to data directory
data_dir = 'data'

# Get all jpg files
jpg_files = [f for f in os.listdir(data_dir) if f.endswith('.jpg')]

# Process each jpg file
results = {}
for jpg_file in jpg_files:
    # Full path to the image
    img_path = os.path.join(data_dir, jpg_file)
    
    # Read the image
    print(f"Processing {jpg_file}...")
    
    # Use easyOCR to detect text
    text_results = reader.readtext(
        img_path,
        decoder = 'greedy',  # 'greedy', 'beamsearch' and 'wordbeamsearch'.
        mag_ratio=0.5,
        batch_size = 1,
        # paragraph = True
        )
    
    # Store results
    results[jpg_file] = text_results
    
    # Display the image and detected text
    img = Image.open(img_path)
    plt.figure(figsize=(10, 10))
    plt.imshow(np.array(img))
    
    # Draw bounding boxes for each detected text
    if text_results:
        for detection in text_results:
            bbox, text, score = detection
            # Convert bbox coordinates to numpy array
            bbox = np.array(bbox)
            
            # Draw rectangle
            plt.plot([bbox[0][0], bbox[1][0], bbox[2][0], bbox[3][0], bbox[0][0]], 
                    [bbox[0][1], bbox[1][1], bbox[2][1], bbox[3][1], bbox[0][1]], 
                    'r-', linewidth=2)
            
            # Add text label with confidence score
            plt.text(bbox[0][0], bbox[0][1]-10, f"{text} ({score:.2f})", 
                    color='red', fontsize=12, bbox=dict(facecolor='white', alpha=0.7))
    
    plt.title(f"Image: {jpg_file}")
    plt.axis('off')
    plt.show()
    
    # Print detected text
    print(f"Text detected in {jpg_file}:")
    if text_results:
        for detection in text_results:
            bbox, text, score = detection
            print(f"Text: {text}, Confidence: {score:.2f}")
            print(f"Bounding Box: {bbox}")
    else:
        print("No text detected")
    print("-" * 50)

print("Processing complete!")

ModuleNotFoundError: No module named 'easyocr'

In [32]:
if text_results:
    for detection in text_results:
        bbox, text, score = detection
        print(f"Text: {text}, Confidence: {score:.2f}")
        print(f"Bounding Box: {bbox}")
else:
    print("No text detected")

No text detected


In [1]:
import os
import easyocr
import numpy as np
from PIL import Image
import cv2

dir = 'data'
images = [Image.open(os.path.join(dir, f)) 
          for f in os.listdir(dir) if f.endswith('.jpg')]

images = [np.array(Image.open(os.path.join(dir, f))) 
          for f in os.listdir(dir) if f.endswith('.jpg')]


# images = [os.path.join(dir, f) 
#           for f in os.listdir(dir) if f.endswith('.jpg')]

# images = [cv2.imread(os.path.join(dir, f)) 
#           for f in os.listdir(dir) if f.endswith('.jpg')]

reader = easyocr.Reader(['en'])

images

/home/fbernardi/miniconda3/envs/text-in-images/lib/python3.10/site-packages/torch/cuda/__init__.py:174: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0
Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


[array([[[255, 242, 204],
         [255, 242, 204],
         [255, 242, 204],
         ...,
         [255, 242, 204],
         [255, 242, 204],
         [255, 242, 204]],
 
        [[255, 242, 204],
         [255, 242, 204],
         [255, 242, 204],
         ...,
         [255, 242, 204],
         [255, 242, 204],
         [255, 242, 204]],
 
        [[255, 242, 204],
         [255, 242, 204],
         [255, 242, 204],
         ...,
         [255, 242, 204],
         [255, 242, 204],
         [255, 242, 204]],
 
        ...,
 
        [[255, 242, 204],
         [255, 242, 204],
         [255, 242, 204],
         ...,
         [255, 242, 204],
         [255, 242, 204],
         [255, 242, 204]],
 
        [[255, 242, 204],
         [255, 242, 204],
         [255, 242, 204],
         ...,
         [255, 242, 204],
         [255, 242, 204],
         [255, 242, 204]],
 
        [[255, 242, 204],
         [255, 242, 204],
         [255, 242, 204],
         ...,
         [255, 242, 204],
  

In [2]:
text_results = reader.readtext_batched(
        images,
        n_width=1024*2,
        n_height=1024*2,
        decoder = 'greedy',
        mag_ratio=0.5,
        batch_size = 3,
        paragraph = True
        )

/home/fbernardi/miniconda3/envs/text-in-images/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [15]:
type(text_results)

list

In [17]:
len(text_results)

7

In [5]:
# Create a boolean array indicating which images contain text
# True if the corresponding element in text_results has length > 0
import numpy as np

# Convert to numpy array for vectorized operation without list comprehension
text_results_bool = [len(result) > 0 for result in text_results]

print("Images containing text:", text_results_bool)
print("Number of images with text:", np.sum(text_results_bool))

Images containing text: [True, False, True, True, True, True, False]
Number of images with text: 5


In [18]:
import torch
import gc

# Optional: Force garbage collection
gc.collect()

# Clear PyTorch CUDA cache if available
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("Cleared PyTorch GPU cache.")
else:
    print("CUDA not available. No GPU cache to clear.")



CUDA not available. No GPU cache to clear.


In [9]:
text_results_bool = [len(result) > 0 for result in text_results]
print(text_results_bool)

[True, False, True, True, True, True, False]
